In [2]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [5]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


# Link -> https://datapathsala.com/problems/product-sales-first-year?difficulty=medium

# Problem: Product Sales in First Year

## Problem Statement

You are given two DataFrames:

### **sales**

Contains sales transaction records with the following columns:

| Column | Data Type | Description |
|---------|-----------|-------------|
| `sale_id` | int | Unique sale identifier |
| `product_id` | int | Product identifier |
| `year` | int | Year of the sale |
| `quantity` | int | Quantity sold |
| `price` | float | Unit price |

### **product**

Contains product information with the following columns:

| Column | Data Type | Description |
|---------|-----------|-------------|
| `product_id` | int | Unique product identifier |
| `product_name` | str | Name of the product |

---

## Task

For each product, find **all sales records** (`quantity` and `price`) from the **first year** the product was sold.

- If a product has **multiple sales** in its first year, include **all** of those records.
- Return the results sorted by:
  1. `product_name` in **ascending** order.
  2. For products with multiple first-year sales, preserve the order by `sale_id` in **ascending** order.

---

## Input

### sales

| sale_id | product_id | year | quantity | price |
|---------:|-----------:|-----:|---------:|------:|
| 1 | 1 | 2020 | 100 | 9.99 |
| 2 | 1 | 2020 | 50 | 10.99 |
| 3 | 1 | 2021 | 200 | 8.99 |
| 4 | 2 | 2019 | 50 | 29.99 |
| 5 | 2 | 2020 | 80 | 27.99 |
| 6 | 3 | 2021 | 30 | 49.99 |
| 7 | 3 | 2022 | 45 | 44.99 |

### product

| product_id | product_name |
|-----------:|--------------|
| 1 | Widget |
| 2 | Gadget |
| 3 | Doohickey |

---

## Expected Output

| product_name | year | quantity | price |
|--------------|-----:|---------:|------:|
| Doohickey | 2021 | 30 | 49.99 |
| Gadget | 2019 | 50 | 29.99 |
| Widget | 2020 | 100 | 9.99 |
| Widget | 2020 | 50 | 10.99 |	29.99
Widget	2020	100	9.99
Widget	2020	50	10.99

# Generating Dataset

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define schema for sales table
sales_schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True)
])

# Data rows for sales
sales_data = [
    (1, 1, 2020, 100, 9.99),
    (2, 1, 2020, 50, 10.99),
    (3, 1, 2021, 200, 8.99),
    (4, 2, 2019, 50, 29.99),
    (5, 2, 2020, 80, 27.99),
    (6, 3, 2021, 30, 49.99),
    (7, 3, 2022, 45, 44.99)
]

# Create sales DataFrame
sales_df = spark.createDataFrame(sales_data, schema=sales_schema)

# Define schema for product table
product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True)
])

# Data rows for product
product_data = [
    (1, "Widget"),
    (2, "Gadget"),
    (3, "Doohickey")
]

# Create product DataFrame
product_df = spark.createDataFrame(product_data, schema=product_schema)

# Show the DataFrames (optional)
sales_df.show()
product_df.show()

+-------+----------+----+--------+-----+
|sale_id|product_id|year|quantity|price|
+-------+----------+----+--------+-----+
|      1|         1|2020|     100| 9.99|
|      2|         1|2020|      50|10.99|
|      3|         1|2021|     200| 8.99|
|      4|         2|2019|      50|29.99|
|      5|         2|2020|      80|27.99|
|      6|         3|2021|      30|49.99|
|      7|         3|2022|      45|44.99|
+-------+----------+----+--------+-----+

+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|      Widget|
|         2|      Gadget|
|         3|   Doohickey|
+----------+------------+



In [9]:
# creating views for SQL query 

sales_df.createOrReplaceTempView("sales")
product_df.createOrReplaceTempView("products")

# Using Spark SQL

In [18]:
spark.sql(
    """WITH first_sale AS (
    SELECT 
        product_id,
        MIN(year) AS first_year
    FROM sales
    GROUP BY product_id
)
SELECT 
    p.product_name,
    s.year,
    s.quantity,
    s.price
FROM sales s
JOIN first_sale f 
    ON f.product_id = s.product_id AND s.year = f.first_year
JOIN products p 
    ON s.product_id = p.product_id
ORDER BY p.product_name ASC, s.sale_id ASC;""").show()

+------------+----+--------+-----+
|product_name|year|quantity|price|
+------------+----+--------+-----+
|   Doohickey|2021|      30|49.99|
|      Gadget|2019|      50|29.99|
|      Widget|2020|     100| 9.99|
|      Widget|2020|      50|10.99|
+------------+----+--------+-----+



# Using Pyspark

In [19]:
from pyspark.sql import Window
from pyspark.sql.functions import min as min_, col

window_spec = Window.partitionBy("product_id")
sales_with_first = sales_df.withColumn("first_year", min_("year").over(window_spec))
result = sales_with_first.filter(col("year") == col("first_year")) \
    .join(product_df, "product_id") \
    .select("product_name", "year", "quantity", "price") \
    .orderBy("product_name", "sale_id")